# Practical Session: LLM Tools – Agentic Behaviour with LangGraph

Welcome to this practical session at RANLP 2025. Over the next ninety minutes we will work together to build a small agentic system using **LangGraph**, a library that orchestrates long‑running workflows across language models and tools. To begin with we will see what a single model can do when asked to summarise a meeting transcript. Then we will gradually introduce structure: breaking the problem into multiple steps, adding loops and branching, and persisting state. LangGraph’s low‑level design makes it possible to checkpoint and resume long‑running agents and to weave in human feedback when needed. By the end of the session you will have a working prototype that reads a meeting transcript and produces a concise report with tasks and decisions.

In [ ]:
#%pip uninstall -q -y torch torchvision torchaudio transformers vllm datasets evaluate sacrebleu sentencepiece langgraph accelerate peft bitsandbytes pandas compressed-tensors spacy langcodes xgrammar vllm

In [ ]:
#%pip install -q torch==2.7.1 torchaudio torchvision --index-url https://download.pytorch.org/whl/cu126
#%pip install -q transformers datasets evaluate sacrebleu sentencepiece langgraph langchain langchain-community grandalf sentence-transformers langchain_huggingface langchain_openai accelerate>=0.21.0 peft bitsandbytes pandas


In [ ]:
# Install Ollama (official script)
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background and save PID (HTTP API on :11434)
!OLLAMA_HOST=0.0.0.0 OLLAMA_ORIGINS="*" nohup ollama serve > ollama.log 2>&1 & echo $! > ollama.pid

# Pull a fast Qwen model + an embedding model
!ollama pull qwen3:8b
!ollama pull nomic-embed-text

# Quick readiness check
!bash -lc 'for i in {1..30}; do curl -s http://localhost:11434/api/tags >/dev/null && { echo "✅ Ollama ready"; exit 0; }; sleep 1; done; echo "⚠️ timeout (see ollama.log)"; exit 1'


⚠️ startup timeout (see llm.log / embed.log)


In [ ]:
import os
import sys
import random
import numpy as np
import torch

# Set a fixed seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Detect device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

Device: cuda



## Problem setting: meeting minutes

Think back to your last research meeting or project stand‑up. Several people talk at once, ideas overlap and the conversation drifts before circling back. Important decisions and follow‑up tasks are easy to miss. In this practical we will design a helper that listens to these meetings and produces a clear summary and a list of tasks with assignees and deadlines. Given a raw transcript, our system will extract the salient points and update a shared task list so everyone knows what to do next.


In the previous practical you saw that a strong foundation model can perform surprisingly well on a range of tasks without fine‑tuning. We'll start by using **gpt‑oss‑20b**, a 20‑billion‑parameter model served by Fireworks, to see how far we can get with a simple prompt. We'll ask it to summarise a meeting transcript and extract action items. Examining the results will show us the strengths and weaknesses of a single LLM call and motivate a more structured approach.

### Experiment: direct prompting

Take a look at the transcript provided in the next cell. It captures a short meeting among three colleagues and includes small talk, interruptions and overlapping topics to mimic the messy reality of meetings. We'll use this example to test our prompt and see how the base model handles summarisation and task extraction.

In [2]:
transcript = """
A: Good morning, everyone. Thanks for joining on short notice. I wanted us to sync about the product launch timeline because we`ve been getting questions from marketing.
B: Morning! Yeah, I saw the emails yesterday. They`re pushing for a concrete date, but I don`t think engineering is fully comfortable committing yet.
C: Exactly. We`re still ironing out some of the backend issues. The integration with the payment system isn`t as smooth as it should be, and if we push too quickly, we`ll have failures during checkout.
A: Right, and we don`t want customers to be the ones finding those bugs. B, from your side, do you think another two weeks would make a difference?
B: Two weeks sounds reasonable, but it depends on how quickly C`s team can finalize the fixes. If QA doesn`t get enough time, we`ll just be pushing the risk down the road.
C: True, but we`ve made progress. Yesterday the team managed to cut down the error rate by almost 40%. If we keep that pace, by the end of next week we should be ready for a full round of regression tests.
A: That`s encouraging. So maybe we can give marketing a tentative date, but with the condition that it`s dependent on QA sign-off.
B: I think they`ll accept that as long as we word it clearly. They mainly need something to build their campaigns around.
C: Should we aim for the 15th, then? That gives us a week to stabilize and a week for QA.
A: Yes, let`s go with that. And if anything unexpected comes up, we`ll update. B, could you draft a short note to marketing after this call?
B: Sure, I`ll take care of it. Do you want me to include the “tentative” language or phrase it as a “target date”?
A: Better to say “target date,” but add that it`s subject to final QA approval. That way we`re transparent without sounding uncertain.
C: Works for me. I`ll also send a daily progress update to both of you so we can react quickly if we see delays.
B: Perfect. One last thing—A, are we still on for the client demo next Thursday?
A: Yes, but let`s keep it internal features only. Nothing related to payments until we`re sure it`s solid.
C: Good call. I`ll make sure the demo build is clean by Wednesday afternoon.
A: Great. Thanks, both of you. I think we`ve got a clear plan now.
B: Sounds good. Talk soon.
C: Bye, everyone.
"""


Let's run our baseline prompt on the short transcript and see what the model produces. Pay attention to whether the summary captures the key points and whether the extracted tasks have clear owners and deadlines.

For these experiments we'll use the **LangChain** framework. **LangChain** provides a unified interface to different LLM providers, utilities for prompt templating and output parsing, and a simple way to chain components together. In the code you'll see we create a prompt template, call the Fireworks model and inspect the results.

> A note on **LangChain**: **LangChain** gives us small building blocks—prompt templates, model wrappers, and a common “runnable” interface—so we can wire prompts to models without hard-coding strings or vendor-specific code. 

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace, HuggingFaceEmbeddings

model_id = "Qwen/Qwen2.5-14B-Instruct"

tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype="bfloat16")  # 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb,
    dtype="bfloat16",
    low_cpu_mem_usage=True,
    device_map="auto",
    do_sample=False,
    temperature=0
)

# IMPORTANT: set return_full_text=False so the prompt isn’t echoed back in the output
gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tok,
    max_new_tokens=8192,
    do_sample=True,
    pad_token_id=tok.eos_token_id,
    return_full_text=False,  # without this, the full prompt is prepended
)

llm_raw = HuggingFacePipeline(pipeline=gen)
llm = ChatHuggingFace(llm=llm_raw)

embed_id = "Qwen/Qwen3-Embedding-0.6B"
embed = HuggingFaceEmbeddings(
    model_name=embed_id,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
    multi_process=False
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Device set to use cuda:0


I'd be happy to help summarize the transcript, but I need the content of the transcript first. Please provide the text you want summarized.
cosine(similar tasks) ≈ 0.864


In [ ]:
system_prompt = "You are a planning assistant in charge of analysing meeting transcripts to extract key information and tasks accurately."
user_prompt = """

Analyze the meeting transcript:

[Transcription]
{transcript}
[End Transcription]

Extract key information and tasks. Your answer should be:

1 - Meeting notes summary (a summary of work related content discussed in the meeting).
2 - Tasks (specific tasks assigned to individuals or teams).
3 - Decisions made (any important decisions or agreements reached during the meeting).
4 - Updates on past tasks (status updates on previously assigned tasks).
"""

`ChatPromptTemplate` lets us declare a sequence of chat messages—usually a `"system"` instruction plus a `"human"` message with placeholders like `{transcript}`—and render them into exactly what the model expects. It’s safer (no brittle string concatenation), easier to read, and it stays portable if we switch models later. You can think of it as a **form** where the transcript is the field we fill in; LangChain handles the formatting.


In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", user_prompt),
])

LangChain introduces a concise syntax called the LangChain Expression Language (LCEL) that lets you compose *runnables* using a pipe operator (`|`). For example:

```python
(prompt | llm).invoke({"transcript": transcript_short})
```

first formats the input with the prompt and then passes the result to the model. This is analogous to Unix pipes: the output of one stage becomes the input to the next. Using `|` keeps our code clean and supports streaming and easy debugging.

In [6]:
response = (prompt | llm).invoke({"transcript": transcript})
print("Response:\n", response.content)

Response:
 **1 – Meeting Notes Summary**  
The team reviewed the product launch timeline in response to marketing’s push for a concrete date. Engineering is still ironing out backend integration issues with the payment system, which could lead to checkout failures if rushed. Progress has been made—error rates were cut by ~40%, and the team expects to finish remaining fixes by the end of next week, followed by a full regression test cycle. The group agreed to offer marketing a *target* launch date of **15 th** conditional on QA sign‑off. Marketing will be informed with clear “target date” language that the date is subject to final QA approval. A client demo is scheduled for next Thursday; it will showcase only internal features, omitting payment‑related functionality until the payment module is stable. Daily progress updates will be sent to keep all parties informed.

**2 – Tasks**  

| Assignee | Task | Deadline / Notes |
|----------|------|------------------|
| **B (Engineering Lead)*

Take a moment to read the model's output. Does the summary reflect the main themes of the meeting? Are the tasks clearly identified with owners and due dates? Make note of any omissions or hallucinations—you'll use these observations to improve the pipeline in the next sections.

## Building a basic meeting‑minutes agent

In the previous section you saw how a single prompt can summarise a meeting and extract tasks. However, relying on one prompt is brittle: important details are missed and tasks may be misassigned. In this stage we’ll construct a basic agent with LangGraph that decomposes the job into three nodes – one to summarise the transcript, one to extract tasks, and one to list explicit decisions. By structuring the workflow we’ll make each step simpler and easier to debug.

Before we can build our agent we need to define the state that flows through it. 

LangGraph uses *schemas* to describe the structure of the state. The main kinds of schemas you can provide to `StateGraph` are:

- **state\_schema** – describes the complete set of keys your graph may read or write. It can be a `TypedDict`, a `dataclass` or a Pydantic `BaseModel`.
- **input\_schema** – a subset of the state schema specifying which keys must be provided when you call `invoke`. When omitted, the state schema serves as the input schema.
- **output\_schema** – a subset of the state schema defining which keys are returned from the graph. Use this to hide internal channels or intermediate values.
- **context\_schema** – an optional read‑only structure passed to every node, used for immutable configuration or dependencies such as database connections or, in our case, the `call\_vllm` helper.

You can also define private state classes for channels that are only used for internal communication between nodes.

#### Step 1 - Summarisation Node

For the first step the state only needs two fields: the original transcript and a summary that we’ll populate.

In [7]:
# Minimal LangGraph imports
from langgraph.graph import StateGraph, END, START
from typing_extensions import TypedDict

# Our minimal state holds the meeting transcript and a summary.
class OverallState(TypedDict):
    transcript: str
    summary: str

# Define which keys are required as input when invoking the graph.
class InputState(TypedDict):
    transcript: str

# Define which keys we want to expose at the end of the graph.
class OutputState(TypedDict):
    summary: str


We then write a summarize_node function that uses our chat model to produce a concise summary of the meeting. This node will read the transcript from the state and return a new dictionary with the summary.

In [8]:
# Node: generate a concise summary of the meeting transcript using our LLM helper.
def summarize_node(state: InputState) -> dict:
    system_prompt = (
        "You are a planning assistant who writes very concise summaries of meetings. "
        "Capture only the most important objectives and outcomes."
    )
    user_prompt = "Summarize the following meeting transcript in a few sentences:{transcript}"
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", user_prompt),
    ])

    response = (prompt | llm).invoke({"transcript": state["transcript"]})
    return {"summary": response.content}


Now, using LangGraph simple abstractions we define the different nodes and connect them. 

In [9]:
# Build a minimal graph that just summarizes the transcript
builder = StateGraph(OverallState, input_schema=InputState, output_schema=OutputState)
builder.add_node('summarize', summarize_node)
builder.set_entry_point('summarize')
builder.add_edge('summarize', END)

# Compile the graph
summary_graph = builder.compile()


Once the graph schema is completed, we compile it and invoke our agent passing the transcript in the **InputState**.

In [10]:
# Invoke the graph on our short transcript
result = summary_graph.invoke({"transcript": transcript})
print('Summary:', result['summary'])

Summary: The team met to lock down a product‑launch date after marketing pushed for a concrete timeline. Engineering noted that backend‑payment integration still had bugs and QA needed more time, but recent progress (40 % error‑rate drop) allows a realistic target of **15 April**—with the caveat that the date is contingent on final QA sign‑off. **Action items:** B will send marketing a “target date” note; C will issue daily progress updates and prepare the demo build; A will keep the upcoming client demo internal‑feature‑only until payments are stable. The plan is clear and the next steps are assigned.


#### Step 2 - Extracting Tasks

A summary gives us the high‑level picture, but we also need to know what work needs to be done. We extend our state to include a `tasks` field and write a `tasks_node` function. This node asks the model to list each action item along with the person responsible and any deadlines.

In [11]:
# Extended state with tasks
class OverallState(TypedDict):
    transcript: str
    summary: str
    tasks: str

# Node: extract tasks
def tasks_node(state: OverallState) -> dict:
    system_prompt = (
        "You are a planning assistant who extracts tasks from meetings. "
        "List each task with the responsible person and any deadlines. Use bullet points."
    )
    user_prompt = "Extract the tasks from this meeting transcript:{transcript}"
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", user_prompt),
    ])

    response = (prompt | llm).invoke({"transcript": state["transcript"]})
    return {"tasks": response.content}


In [12]:
# Build a graph with two sequential nodes: summarization followed by task extraction
builder = StateGraph(OverallState)
builder.add_node('summarize', summarize_node)
builder.add_node('tasks', tasks_node)
builder.set_entry_point('summarize')
builder.add_edge('summarize', 'tasks')
builder.add_edge('tasks', END)

meeting_graph = builder.compile()

result = meeting_graph.invoke({"transcript": transcript})
print('Summary:', result['summary'])
print('Tasks:', result['tasks'])


Summary: The team met to align on a product launch date after marketing requested a concrete timeline. Engineers explained that backend payment integration still has bugs, and QA needs time to verify fixes, so the group settled on a target launch date of the 15th, contingent on final QA approval. A will provide marketing with a note that frames the date as a “target” pending QA sign‑off, while B will draft the communication. C will send daily progress updates and ensure a clean demo build (restricted to internal features only) is ready by Wednesday for the client demo next Thursday. The plan is now clear, with agreed next steps for communication and quality assurance.
Tasks: - **Draft & send marketing note** – **B**  
  - Include “target date” of the 15th with a caveat that it’s subject to final QA approval.  
  - **Deadline:** Immediately after the call (same day).  

- **Daily progress updates** – **C**  
  - Send a brief status email to A and B each day so delays can be caught early

### Step 3: Capture Decisions

Meetings often end with explicit decisions or agreements. To capture these, we add a `decisions` field to our state and write a `decisions_node`. This node lists the important decisions made during the meeting in bullet form. We then combine our three nodes into a single graph to produce a structured report.

In [13]:
# Extend the state again to include decisions
class OverallState(TypedDict):
    transcript: str
    summary: str
    tasks: str
    decisions: str

def decisions_node(state: OverallState) -> dict:
    system_prompt = (
        "You are a planning assistant who lists important decisions made during a meeting. "
        "Return bullet points summarising each decision."
    )
    user_prompt = "List any explicit decisions from this meeting transcript:{transcript}"
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", user_prompt),
    ])

    response = (prompt | llm).invoke({"transcript": state["transcript"]})
    
    return {"decisions": response.content}


In [14]:
# Build a graph with three nodes: summarize -> tasks -> decisions
builder = StateGraph(OverallState)
builder.add_node('summarize', summarize_node)
builder.add_node('tasks', tasks_node)
builder.add_node('decisions', decisions_node)
builder.set_entry_point('summarize')
builder.add_edge('summarize', 'tasks')
builder.add_edge('tasks', 'decisions')
builder.add_edge('decisions', END)

full_graph = builder.compile()

result = full_graph.invoke({"transcript": transcript})
print('Summary:', result['summary'], end='\n\n')
print('Tasks:', result['tasks'], end='\n\n')
print('Decisions:', result['decisions'])


Summary: **Meeting Summary**

- **Goal:** Finalize a launch date for the product while addressing backend and payment‑integration bugs.  
- **Outcome:** Agree on a **target launch date of August 15**, contingent on full QA sign‑off.  
- **Action Items:**  
  - *B* will draft a marketing note using “target date” wording with a QA‑approval caveat.  
  - *C* will send daily progress updates and ensure the demo build is clean by Wednesday.  
  - *A* will keep next Thursday’s client demo limited to internal features, postponing payment‑related demos until stability is confirmed.

Tasks: - **B** – Draft and send a short note to marketing that specifies the **target launch date of the 15th** and notes it is “subject to final QA approval.”  
  *Deadline: within the next 24 hours (after the meeting).*

- **C (Engineering/QA team)** – Perform a full round of regression tests and obtain sign‑off **before the 15th.**  
  *Deadline: regression testing complete by the end of next week; sign‑off by t

Our basic agent successfully decomposes the meeting summarisation problem into three steps: summarising the transcript, extracting tasks, and listing decisions. This simple graph demonstrates how LangGraph passes state between nodes and makes each step independent. However, the tasks are free‑form strings, meaning that some items might be missing, others could be ambiguous, and there is no way to validate them automatically. To build a more robust assistant we need to validate and refine the model’s outputs. In the next stage we will add a self‑reflection loop.

## 🧪 Exercise 1: Add a "highlights" node after the summary

**What:** From the existing `summary` in state, create 2–3 concise bullet highlights.  
**Why:** Practice adding a node that *derives* data from previous state, no LLM needed.

**Rules:**
- Use simple heuristics (split by sentences, pick the 2–3 most informative).
- Return `{"highlights": List[str]}`.
- Wire this node **after** `summary`.

**Success check:** After running the graph, `state["highlights"]` exists and is a short list.


In [15]:
# Your code for the graph here

### Self‑reflection to improve task extraction

In the basic agent we asked the model to list “tasks” and hoped for the best. Sometimes it works; other times it could miss items or invent details. We’re going to teach our agent to check its own work. The idea is simple: generate a first pass of tasks, run a small critic against a rubric (“don’t invent names or dates; keep titles concise; owners must appear in the transcript”), and if it fails, run a targeted repair pass before we accept the result. This follows the “reflection / Reflexion” pattern popular in agent design and is a natural fit for LangGraph because we can route on a condition and even loop a couple of times.

#### Step 1 - Standardise the task format

We’ll switch from free-form task text to **typed objects**. That makes the critic’s job clear and the repair step easier to target. We use Pydantic models for `Task`, `TaskList`, and a tiny `Reflection` result. LangChain supports structured output via schema-aware prompts and parsers; we’ll use a Pydantic parser so this works with any chat model.

We first define what a *Task* is. Binding this schema to the LLM makes the output predictable.

Here we ask the model to return tasks as objects, then normalise and de-duplicate them with Pydantic v2. We accept natural phrasing for due_date (“next Friday”, “Q4”) and drop unknown fields so downstream code stays stable. The model may call fields by different names, so we use AliasChoices to map common variants (e.g., owner, assigned_to, who → assignee). Finally, a light model_validator filters obviously broken items and a pass of de-duplication removes near-duplicates.

In [16]:
from typing import Optional, List, Literal, Dict, Any
from pydantic import BaseModel, Field, ConfigDict, field_validator, AliasChoices, model_validator
import json

Status = Literal["ToDo", "InProgress", "Done", "Closed"]

class Task(BaseModel):
    model_config = ConfigDict(extra="ignore")  # drop unexpected fields
    title: str = Field(min_length=3, validation_alias=AliasChoices("title", "task", "name"))
    assignee: str = Field(min_length=1, validation_alias=AliasChoices("assignee", "person", "owner", "assigned_to", "who"))
    description: str = Field(min_length=1, validation_alias=AliasChoices("description", "details", "info", "notes"))
    # due_date intentionally free text; NO ISO enforcement (can be "next week", "Q4", etc.)
    due_date: Optional[str] = Field(default=None, validation_alias=AliasChoices("due_date", "deadline", "when", "by", "due"))
    status: Status = "ToDo"

    @field_validator("title", "assignee", "description", "due_date", mode="before")
    @classmethod
    def _strip(cls, v):
        return v.strip() if isinstance(v, str) else v

class TaskList(BaseModel):
    tasks: List[Task]
    
    @model_validator(mode="before")
    @classmethod
    def _normalize_and_filter(cls, v):
        """Accept various wrapper shapes and drop blatantly invalid task dicts early."""
        items = v if isinstance(v, list) else v.get("tasks") if isinstance(v, dict) else None
        if items is None and isinstance(v, dict) and isinstance(v.get("properties"), dict):
            maybe = v["properties"].get("tasks")
            if isinstance(maybe, list):
                items = maybe
        if items is None:
            return v  # let Pydantic raise later if shape invalid

        filtered = []
        for it in items:
            if not isinstance(it, dict):
                continue
            title = (it.get("title") or it.get("task") or it.get("name") or "").strip()
            assignee = (it.get("assignee") or it.get("person") or it.get("owner") or it.get("assigned_to") or it.get("who") or "").strip()
            desc = (it.get("description") or it.get("details") or it.get("info") or it.get("notes") or "").strip()
            if title and assignee and desc:
                filtered.append(it)
        return {"tasks": filtered}

    @model_validator(mode="after")
    def _deduplicate(self):
        """Deduplicate tasks (case-insensitive on key fields) preserving first occurrence."""
        seen = set()
        dedup = []
        for t in self.tasks:
            key = (
                t.title.strip().lower(),
                t.assignee.strip().lower(),
                t.description.strip().lower(),
                (t.due_date or "").strip().lower(),
                t.status,
            )
            if key not in seen:
                seen.add(key)
                dedup.append(t)
        self.tasks = dedup
        return self

Now, our new state should hold the `TaskList` the model is going to output

In [17]:

class OverallState(TypedDict):
    transcript: str
    summary: str
    decisions: str
    tasks_struct: TaskList  # validated list of Task dicts for downstream automation

LangChain’s `with_structured_output()` binds our Pydantic model to the chat model so the return value is already parsed and validated. We still give the model a short rubric to reduce guesswork. The prompt is assembled with ChatPromptTemplate, which keeps variables and messaging tidy.

> Note: You must verify if your model allows this. This is a feature embedded into the LLM and allowed by the AI Provider.

In [18]:
tasklist_llm = llm.with_structured_output(
    TaskList, 
    method="json_schema"
)

The `tasks_structured_node` node reads the meeting `transcript` and returns a validated `TaskList`.

In [19]:
SCHEMA_CONSTRAINTS = """
    Schema & constraints:
    - Only keys: title, description, assignee, due_date?, status.
    - Title and Description are non-empty strings must be representative of the task at hand.
    - Status ∈ ["ToDo", "InProgress", "Done", "Closed"] (exact casing).
    - due_date is optional (null of free text); keep whatever natural phrasing appears (e.g. "next Friday", "in two weeks", "Q4"). Do NOT normalize to ISO.
    - Assignee must be a non-empty string, is a speaker in the transcript.
"""

def tasks_structured_node(state: OverallState) -> Dict:
    system_prompt = f"""
        You are a planning assistant who extracts tasks from meeting transcripts.
        Your goal is to identify and extract actionable tasks from a transcript.
        Return ONLY a TaskList JSON *instance* (the data), not a JSON Schema.
        
        {SCHEMA_CONSTRAINTS}
    """
    
    user_prompt = "Transcript:\n{transcript}\nYour answer must be a JSON object."
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", user_prompt),
    ])
    
    tasklist = (prompt | tasklist_llm).invoke({"transcript": state["transcript"]})
    return {"tasks_struct": tasklist}

We insert our new tasks_structured node between summarize and decisions. This keeps the flow readable and lets each node focus on one job. The code below reuses summarize_node and decisions_node from Stage 1; run those cells first if you skipped ahead.

In [20]:
import pprint

# Build a graph with three nodes: summarize -> tasks -> decisions
builder = StateGraph(OverallState)
builder.add_node('summarize', summarize_node)
builder.add_node('tasks_structured', tasks_structured_node)
builder.add_node('decisions', decisions_node)
builder.set_entry_point('summarize')
builder.add_edge('summarize', 'tasks_structured')
builder.add_edge('tasks_structured', 'decisions')
builder.add_edge('decisions', END)

full_graph = builder.compile()


In [21]:

result = full_graph.invoke({"transcript": transcript})
print('Summary:', result['summary'], end='\n\n')
print('Tasks:')

for t in result['tasks_struct']:
    pprint.pprint(t)
    print()
print()
print('Decisions:', result['decisions'])


Summary: **Meeting Summary – Product Launch & Demo Coordination**

- **Launch Target:** Set the tentative launch date for the 15th, contingent on final QA sign‑off.  
- **Marketing Communication:** B will draft a note to marketing, describing the date as a “target” with a caveat that it depends on QA approval.  
- **QA & Backend Progress:** C’s team has reduced payment‑system errors by ~40% and aims to complete regression testing by the end of next week; daily progress updates will be sent to A and B.  
- **Client Demo:** The demo for next Thursday will focus only on internal features; C will deliver a clean build by Wednesday afternoon, avoiding any payment‑related functionality until it’s verified.  

This plan aligns engineering capacity with marketing’s timeline needs while keeping stakeholders informed on risk and progress.

Tasks:
('tasks',
 [Task(title='Draft email to marketing', assignee='B', description='Draft a short note to marketing outlining the tentative launch date, noti

Great! now we have proper task extraction. Run the Cell above again a few times, you might see that once in a while, there are missalignments in the task detection. Sometimes it detects 3 tasks. Other times it detects 4, and so on.

This is how our graph looks right now.

In [22]:
# from IPython.display import Image, display
# display(Image(full_graph.get_graph().draw_mermaid_png()))

print(full_graph.get_graph().draw_ascii())

    +-----------+    
    | __start__ |    
    +-----------+    
          *          
          *          
          *          
    +-----------+    
    | summarize |    
    +-----------+    
          *          
          *          
          *          
+------------------+ 
| tasks_structured | 
+------------------+ 
          *          
          *          
          *          
    +-----------+    
    | decisions |    
    +-----------+    
          *          
          *          
          *          
    +---------+      
    | __end__ |      
    +---------+      


#### Step 2 - Critique and repair the task list

Even with a schema, the model sometimes omits tasks or includes spurious ones. We can improve reliability by adding a critic. We define a `Reflection` Pydantic model with three fields: `approve` (a boolean indicating whether the current task list meets our rubric), `issues` (a list of short descriptions of problems) and `instructions` (concise guidance on how to fix those problems). A critic node uses this model to decide whether the candidate task list is acceptable. If it isn’t, a repair node generates a revised list following the critic’s instructions. We loop between critic and repair until the tasks are approved or a maximum number of iterations is reached. LangGraph’s conditional edges and immutable state make it easy to express this loop.

In [23]:
MAX_REFLECTION_LOOPS = 3 # keep tiny for speed & determinism

class Reflection(BaseModel):
    approve: bool = Field(..., description="True if the candidate TaskList passes the rubric.")
    issues: List[str] = Field(default_factory=list, description="Concise bullet points of problems found.")
    instructions: str = Field(..., description="Short, actionable instructions to revise the TaskList.")

Our `OverallState` now has to support storing critics and attempt counts

In [24]:
class OverallState(TypedDict):
    transcript: str
    summary: str
    decisions: str
    tasks_struct: TaskList
    approved: bool
    attempts: int
    issues: List[str]
    instructions: List[str]
    history: List[Dict[str, Any]]

We encode the acceptance rubric in the critic’s system prompt, and bind both critic and repair to structured outputs so they return `Reflection` and `TaskList` objects directly.

In [25]:
INSTRUCTIONS_STYLE = """
    - '+ title=... assignee=...': add a grounded task; ensure fields are supported by cited lines.
    - '- title=... assignee=...': remove EXACT matching task if present.
    - '~ title=... assignee=... field=... -> ...': modify the specified field value only.
"""

REFLECTION_SYSTEM = f"""
    You are a strict TASK QUALITY AUDITOR. Review a candidate TaskList against this rubric:

    {SCHEMA_CONSTRAINTS}

    GROUNDING REQUIREMENT:
    For every issue you raise, cite supporting transcript line numbers like L12 or ranges L34-L37. You will receive the transcript as a single block; treat each newline-delimited utterance as one line (first line = L1). If evidence spans non-contiguous lines, list them comma-separated.

    Return a Reflection JSON: approve (bool), issues (list of strings), instructions (string).

    All Instructions MUST be a plan consisting of one directive per line. For example:

    {INSTRUCTIONS_STYLE}

    Reject on any violation but accept clear interpretations that might not be explicit.
"""

REPAIR_SYSTEM = f"""
    You are a TASK REPAIR EXPERT. You will receive a transcript, a candidate list of TASKS and some Instructions to fix the TASKS.

    You must follow ONLY the Instructions and FIX the TASKS accordingly. Treat them atomically:
    {INSTRUCTIONS_STYLE}

    Preserve unchanged tasks.
    If a directive cannot be executed due to missing grounding, SKIP it (do not guess).
    Return ONLY a valid TaskList JSON object (the instance, not a schema).
"""


In [26]:
critic_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", REFLECTION_SYSTEM),
        ("human", 
            (
                "Transcript (for grounding; each line is numbered implicitly by order):\n{transcript}\n\n"\
                "Candidate TaskList JSON:\n{candidate_json}\n\n"\
                "Return the Reflection JSON now."
            ),
        )
    ]
)

repair_prompt = ChatPromptTemplate.from_messages(
    [
    ("system", REPAIR_SYSTEM),
    ("human", 
        (
            "Transcript:\n{transcript}\n\n"\
            "Candidate TaskList JSON (to revise):\n{candidate_json}\n\n"\
            "Diff-style critic instructions:\n{instructions}\n\n"\
            "FOLLOW THE INSTRUCTIONS and Fix the TaskList. Return ONLY the fixed TaskList JSON."
        ),
    )
    ]
)

reflection_llm = llm.with_structured_output(Reflection, 
    method="json_schema"
)

critic_chain = critic_prompt | reflection_llm
repair_chain = repair_prompt | tasklist_llm

Now, our `reflect_node` runs the critic. If the critic approves or we hit the loop cap, we stop; otherwise we store instructions for the repair step.
`repair_node` regenerates a corrected `TaskList` using the critic’s instructions. We also append a small history trail for debugging.

In [27]:
def reflect_node(state: OverallState) -> Dict[str, Any]:
    candidate_json = state["tasks_struct"].model_dump_json()
    attempts = state.get("attempts", 0)
    history = state.get("history", [])

    # Pass raw transcript; critic infers line numbers by order.
    reflection = critic_chain.invoke(
        {
            "transcript": state["transcript"],
            "candidate_json": candidate_json
        }
    )

    reflection_approved = reflection.approve and not reflection.issues
    
    out = {
        "approved": reflection_approved,
        "issues": reflection.issues,
        "history": history + [{"stage": "reflect", **reflection.model_dump()}],
        "attempts": attempts,  # increment on repair
    }

    if reflection_approved or attempts >= MAX_REFLECTION_LOOPS:
        return out
    
    print(f"Reflection found issues (attempt {attempts + 1}):")
    for issue in reflection.issues:
        print(f" - {issue}")
        

    out["instructions"] = reflection.instructions
    return out

In [28]:
def repair_node(state: OverallState) -> Dict[str, Any]:    
    candidate_json = state["tasks_struct"].model_dump_json()  # list[dict]
    attempts = state.get("attempts", 0)
    history = state.get("history", [])

    print(f"Instructions: {state.get('instructions', '')}")

    fixed: TaskList = repair_chain.invoke(
        {
            "transcript": state["transcript"],
            "candidate_json": candidate_json,
            "instructions": state.get("instructions", ""),
        }
    )

    print(f"Repair attempt {attempts + 1} completed. New Tasks: {fixed}")

    return {
        "tasks_struct": fixed,
        "attempts": attempts + 1,
        "history": history + [{"stage": "repair", "tasks": fixed}],
    }

Now, we should wire the loop with a gate

We send control `tasks_structured → reflect → (approve? decisions : repair → reflect …)`.

In [29]:
from langgraph.graph import StateGraph, END

builder = StateGraph(OverallState)
builder.add_node("summarize", summarize_node) # existing
builder.add_node("tasks_structured", tasks_structured_node) # existing
builder.add_node("reflect", reflect_node) # new
builder.add_node("repair", repair_node) # new
builder.add_node("decisions", decisions_node) # existing

builder.set_entry_point("summarize")
builder.add_edge("summarize", "tasks_structured")
builder.add_edge("tasks_structured", "reflect")

def _gate(state: OverallState) -> str:
    # stop if approved or we've reached the small retry cap
    if state.get("approved") or state.get("attempts", 0) >= MAX_REFLECTION_LOOPS:
        return "decisions"
    return "repair"

builder.add_conditional_edges("reflect", _gate, {"repair": "repair", "decisions": "decisions"})
builder.add_edge("repair", "reflect")
builder.add_edge("decisions", END)

full_graph_reflect = builder.compile()

In [30]:
refl_state = full_graph_reflect.invoke({"transcript": transcript})

print("\nRefined tasks after reflection:")
pprint.pprint(refl_state["tasks_struct"].tasks)

print("\nApproved?", refl_state.get("approved"), " | Attempts:", refl_state.get("attempts", 0))
if refl_state.get("issues"):
    print("Critic issues:", *refl_state["issues"], sep="\n - ")


Refined tasks after reflection:
[Task(title='Draft note to marketing', assignee='B', description='Draft a short note to marketing after the call, phrasing the target date with QA approval condition.', due_date='after call', status='ToDo'),
 Task(title='Send daily progress updates', assignee='C', description='Send a daily progress update to A and B to react quickly if delays.', due_date=None, status='ToDo'),
 Task(title='Prepare demo build for demo', assignee='C', description='Make sure the demo build is clean by Wednesday afternoon for the client demo on Thursday.', due_date='Wednesday afternoon', status='ToDo'),
 Task(title='Ensure internal features only demo on Thursday', assignee='A', description='Ensure client demo on Thursday shows only internal features, no payment-related functionality.', due_date='Thursday', status='ToDo')]

Approved? True  | Attempts: 0


This is how our new graph looks like. You can see we have a small cycle following the Reflexion pattern, in which externally from the actual generation, we get an evaluation and instructions to fix/improve the generated tasks. Then, the repair node take those instructions and try to fix the candidate response previously shown.

In [31]:
# display(Image(full_graph_reflect.get_graph().draw_mermaid_png()))
print(full_graph_reflect.get_graph().draw_ascii())

         +-----------+             
         | __start__ |             
         +-----------+             
                *                  
                *                  
                *                  
         +-----------+             
         | summarize |             
         +-----------+             
                *                  
                *                  
                *                  
      +------------------+         
      | tasks_structured |         
      +------------------+         
                *                  
                *                  
                *                  
          +---------+              
          | reflect |              
          +---------+              
          ..         ..            
        ..             ..          
       .                 .         
+--------+          +-----------+  
| repair |          | decisions |  
+--------+          +-----------+  
                          * 

Lets take a peek at the history. Lets see why the gate decided to stop

In [32]:
refl_state

{'transcript': '\nA: Good morning, everyone. Thanks for joining on short notice. I wanted us to sync about the product launch timeline because we`ve been getting questions from marketing.\nB: Morning! Yeah, I saw the emails yesterday. They`re pushing for a concrete date, but I don`t think engineering is fully comfortable committing yet.\nC: Exactly. We`re still ironing out some of the backend issues. The integration with the payment system isn`t as smooth as it should be, and if we push too quickly, we`ll have failures during checkout.\nA: Right, and we don`t want customers to be the ones finding those bugs. B, from your side, do you think another two weeks would make a difference?\nB: Two weeks sounds reasonable, but it depends on how quickly C`s team can finalize the fixes. If QA doesn`t get enough time, we`ll just be pushing the risk down the road.\nC: True, but we`ve made progress. Yesterday the team managed to cut down the error rate by almost 40%. If we keep that pace, by the end

By asking the model to evaluate its own output we improve task quality without changing the base model. The reflection loop demonstrates how LangGraph can route execution based on state and support iterative refinement.

### Persisting tasks with Langgraph Memory

Our agent now produces a clean, approved TaskList. If we stop here, those tasks vanish when the run ends. In this stage we give the agent a simple long-term memory so tasks survive across runs and across transcripts. LangGraph was built for stateful, long-running agents and supports durability and memory out-of-the-box. We’ll keep it simple: a tiny key–value store for tasks, plus a dash of semantic matching so we update existing tasks instead of creating duplicates.

In [33]:

import re
from langchain_fireworks import FireworksEmbeddings  # Fireworks embeddings
from langgraph.store.base import BaseStore           # Type for injection

There are two kinds of memory to keep straight:

- Checkpointing: lets the graph pause/resume during execution (e.g., across the reflection loop). That’s short-term robustness within a run.
- Task storage: a small, persistent store where approved tasks are written so they’re still there on the next run.

LangGraph’s design helps with both: durable execution and memory primitives for agent state. In our class we’ll use a lightweight local store so everyone can run this without extra services. 

To keep the focus on agent logic, we use a in-memory key–value store. Each task is saved under a stable key (we’ll derive it from the `title`) and we keep the original fields (`title`, `description`, `assignee`, `due_date`, `status`).

In [34]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()

And a simple list tasks function to see what we got in store

In [35]:
def _ns(user_id: str) -> tuple[str, ...]:
    return ("users", user_id, "tasks")

def list_tasks_store(store: InMemoryStore, user_id: str, limit: int = 1000) -> List[dict]:
    ns = _ns(user_id)
    # No query ⇒ list; increase limit for classroom demos. :contentReference[oaicite:12]{index=12}
    items = store.search(ns, limit=limit)
    out = []
    for it in items:
        v = it.value
        out.append({
            "key": it.key,
            "title": v.get("title"),
            "description": v.get("description"),
            "assignee": v.get("assignee"),
            "due_date": v.get("due_date"),
            "status": v.get("status"),
            "score": it.score,  # may be None if no query
        })
    return out

Our new state includes the `persist_results`

In [36]:

from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

class OverallState(TypedDict, total=False):
    transcript: str
    summary: str
    decisions: str
    tasks_struct: TaskList
    approved: bool
    attempts: int
    issues: List[str]
    instructions: List[str]
    persist_results: List[dict]
    history: List[Dict[str, Any]]

We define our persist node with key slug normalization

In [37]:
def _slug(s: str) -> str:
    s = "" if s is None else str(s)
    s = re.sub(r"[^a-z0-9\s\-_/]", "", s.strip().lower())
    s = re.sub(r"[\s/]+", "-", s)
    return s[:120] or "untitled"

# Persist ALL tasks deterministically (no tool-calling needed here)
def persist(state: OverallState, *, store: BaseStore):
    user_id = "u-demo"
    task_list = state["tasks_struct"]
    ns = _ns(user_id)

    results = []
    print("Persisting tasks...")
    for task in task_list.tasks:
        key = _slug(task.title)

        key_exists = store.get(ns, key) is not None
        action = "updated" if key_exists else "created"

        store.put(ns, key, task.model_dump())

        results.append({"action": action, "key": key})
        print(f"  ├─ {action.upper():7} → {key} [status={task.status}]")

    return {"persist_results": results}

And we are ready for building our graph again.

After the reflection loop approves the TaskList, the graph runs a persist node that:

- iterates through tasks_struct.tasks,
- looks up a best match in the store (semantic + lexical), and
- upserts (update-or-insert) a single normalized record per task.

Nothing here is model-dependent; it’s just deterministic logic that makes the agent feel like a real application instead of a demo.

In [38]:
# Build your graph as before
builder = StateGraph(OverallState)
builder.add_node("summarize", summarize_node)
builder.add_node("tasks_structured", tasks_structured_node)
builder.add_node("reflect", reflect_node)
builder.add_node("repair", repair_node)
builder.add_node("decisions", decisions_node)
builder.add_node("persist", persist)

builder.set_entry_point("summarize")
builder.set_finish_point("decisions")

def _gate(state: OverallState) -> str:
    if state.get("approved") or state.get("attempts", 0) >= MAX_REFLECTION_LOOPS:
        return "persist"
    return "repair"

builder.add_edge("summarize", "tasks_structured")
builder.add_edge("tasks_structured", "reflect")
builder.add_conditional_edges("reflect", _gate, {"repair": "repair", "persist": "persist"})
builder.add_edge("repair", "reflect")
builder.add_edge("persist", "decisions")
builder.add_edge("decisions", END)

# 3) Compile with the store injected so nodes can receive it
full_graph = builder.compile(store=store)

# display(Image(full_graph_reflect.get_graph().draw_mermaid_png()))
print(full_graph.get_graph().draw_ascii(), end="\n\n")


         +-----------+            
         | __start__ |            
         +-----------+            
               *                  
               *                  
               *                  
         +-----------+            
         | summarize |            
         +-----------+            
               *                  
               *                  
               *                  
     +------------------+         
     | tasks_structured |         
     +------------------+         
               *                  
               *                  
               *                  
          +---------+             
          | reflect |             
          +---------+             
          **        ..            
        **            ..          
       *                .         
+--------+          +---------+   
| repair |          | persist |   
+--------+          +---------+   
                          *       
                    

In [39]:

# 4) Run as usual
final_state = full_graph.invoke({"transcript": transcript})
print("Persist results:", final_state.get("persist_results"))

Persisting tasks...
  ├─ CREATED → draft-marketing-brief-for-launch-date [status=ToDo]
  ├─ CREATED → send-daily-progress-updates [status=InProgress]
  ├─ CREATED → prepare-demo-build-by-wednesday-afternoon [status=ToDo]
Persist results: [{'action': 'created', 'key': 'draft-marketing-brief-for-launch-date'}, {'action': 'created', 'key': 'send-daily-progress-updates'}, {'action': 'created', 'key': 'prepare-demo-build-by-wednesday-afternoon'}]


In [40]:

# Quick peek: list tasks persisted in Store
for row in list_tasks_store(store=store, user_id="u-demo"):
    print(row)

{'key': 'draft-marketing-brief-for-launch-date', 'title': 'Draft marketing brief for launch date', 'description': 'Draft a short note to marketing stating the target date (15th), subject to final QA approval.', 'assignee': 'B', 'due_date': 'right after this call', 'status': 'ToDo', 'score': None}
{'key': 'send-daily-progress-updates', 'title': 'Send daily progress updates', 'description': 'Send a daily progress update to A and B regarding development progress.', 'assignee': 'C', 'due_date': None, 'status': 'InProgress', 'score': None}
{'key': 'prepare-demo-build-by-wednesday-afternoon', 'title': 'Prepare demo build by Wednesday afternoon', 'description': 'Ensure the demo build is clean and includes only internal features, with no payment-related code until stable.', 'assignee': 'C', 'due_date': 'Wednesday afternoon', 'status': 'ToDo', 'score': None}


This short follow‑up transcript simulates a quick sync after the original meeting. We’ll re‑compile the graph with the same store and feed this new text. The goal is to update existing tasks (e.g., “notify marketing about the 15th target date”) rather than creating look‑alike duplicates with slightly different wording. Keep an eye on the ASCII graph printout: it reflects the flow we built—summarise → extract → reflect/repair → persist → decisions.

In [41]:
follow_up_transcript = """
A: Quick sync. First—B, the marketing email about the 15th target date?
B: I have done that, I sent it right after our last call and posted in #marketing. If QA slips, I'll post an update.
C: Noted.
A: Daily progress updates—can we standardize them?
C: Yes. I'll post a short update every day at 17:00 CET in the #launch-updates thread with error rate and blockers. We will keep it in progress.
A: Perfect.
A: What about the Demo build status?
C: Finished Tuesday EOD; it's clean and payments stay disabled. So that task's done.
A: Next Thursday's client demo—let's keep it internal features only. C, can you lead the walkthrough? I'll handle intros and Q&A. Let's set it for 11:00.
C: Works for me.
D (QA): We can kick off full regression Monday and aim to share the report by Wednesday morning.
A: Perfect.
C: To speed triage we need richer checkout error logs; I can take that.
A: Please aim for Friday EOD.
B: After QA sign-off, design wants a one-pager to brief campaigns. I'll draft it Tuesday.
A: And let's have a stakeholder risk review Monday at 09:00—I'll book it.
"""


final_state = full_graph.invoke({"transcript": follow_up_transcript})
print("Persist results:", final_state.get("persist_results"))

Persisting tasks...
  ├─ CREATED → post-daily-progress-updates [status=InProgress]
  ├─ CREATED → lead-client-demo-walkthrough [status=ToDo]
  ├─ CREATED → kick-off-full-regression-test [status=ToDo]
  ├─ CREATED → provide-richer-checkout-error-logs [status=ToDo]
  ├─ CREATED → draft-one-pager-briefing-campaigns [status=ToDo]
  ├─ CREATED → book-stakeholder-risk-review [status=ToDo]
Persist results: [{'action': 'created', 'key': 'post-daily-progress-updates'}, {'action': 'created', 'key': 'lead-client-demo-walkthrough'}, {'action': 'created', 'key': 'kick-off-full-regression-test'}, {'action': 'created', 'key': 'provide-richer-checkout-error-logs'}, {'action': 'created', 'key': 'draft-one-pager-briefing-campaigns'}, {'action': 'created', 'key': 'book-stakeholder-risk-review'}]


In [42]:

# Quick peek: list tasks persisted in Store
for row in list_tasks_store(store=store, user_id="u-demo"):
    print(row)

{'key': 'draft-marketing-brief-for-launch-date', 'title': 'Draft marketing brief for launch date', 'description': 'Draft a short note to marketing stating the target date (15th), subject to final QA approval.', 'assignee': 'B', 'due_date': 'right after this call', 'status': 'ToDo', 'score': None}
{'key': 'send-daily-progress-updates', 'title': 'Send daily progress updates', 'description': 'Send a daily progress update to A and B regarding development progress.', 'assignee': 'C', 'due_date': None, 'status': 'InProgress', 'score': None}
{'key': 'prepare-demo-build-by-wednesday-afternoon', 'title': 'Prepare demo build by Wednesday afternoon', 'description': 'Ensure the demo build is clean and includes only internal features, with no payment-related code until stable.', 'assignee': 'C', 'due_date': 'Wednesday afternoon', 'status': 'ToDo', 'score': None}
{'key': 'post-daily-progress-updates', 'title': 'Post daily progress updates', 'description': 'Post a short daily update at 17:00 CET in t

But we see it repeated some tasks that should be refering to the same thing! Let's fix that with semantic search.


In [43]:
from langchain_fireworks import FireworksEmbeddings

embed = FireworksEmbeddings(model="nomic-ai/nomic-embed-text-v1.5", api_key=API_KEY)  # FIREWORKS_API_KEY env var

store = InMemoryStore(
    index={"dims": 768, "embed": embed, "fields": ["title", "description"]}  # optional
)

USER_ID = "u-demo"

Find a match key (vector search)

In [44]:
# Use the store's vector index (title→embedding) to find nearest neighbor.
SIM_THRESHOLD = 0.7

def _nearest_title(store: BaseStore, ns: tuple, title: str):
    """Return (key, score) for the nearest existing task title, or (None, None)."""
    hits = store.search(ns, query=title, limit=1) or []
    if not hits:
        return None, None
    h = hits[0]
    
    return h.key, getattr(h, "score", None)


Semantic upsert persist node

In [45]:
def persist_semantic(state: OverallState, *, store: BaseStore):
    user_id = USER_ID
    ns = ("users", user_id, "tasks")
    task_list = state.get("tasks_struct")
    results = []
    
    print("Persisting (semantic upsert)…")
    for task in task_list.tasks:
        match_key, score = _nearest_title(store, ns, task.title)
        use_match = (score is not None) and (score >= SIM_THRESHOLD)

        key = match_key if use_match else _slug(task.title)
        action = "updated" if use_match else "created"

        store.put(ns, key, task.model_dump())  # uses the index fields you configured
        results.append({"action": action, "key": key, "score": score})

        print(f"  ├─ {action.upper():7} → {key} [status={task.status}] (score={None if score is None else round(score, 3)})")

    return {"persist_results": results}

In [46]:
# Build your graph as before
builder = StateGraph(OverallState)
builder.add_node("summarize", summarize_node)
builder.add_node("tasks_structured", tasks_structured_node)
builder.add_node("reflect", reflect_node)
builder.add_node("repair", repair_node)
builder.add_node("decisions", decisions_node)
builder.add_node("persist_semantic", persist_semantic)

builder.set_entry_point("summarize")
builder.set_finish_point("decisions")

def _gate(state: OverallState) -> str:
    if state.get("approved") or state.get("attempts", 0) >= MAX_REFLECTION_LOOPS:
        return "persist_semantic"
    return "repair"

builder.add_edge("summarize", "tasks_structured")
builder.add_edge("tasks_structured", "reflect")
builder.add_conditional_edges("reflect", _gate)
builder.add_edge("repair", "reflect")
builder.add_edge("persist_semantic", "decisions")
builder.add_edge("decisions", END)

# 3) Compile with the store injected so nodes can receive it
full_graph = builder.compile(store=store)

# display(Image(full_graph_reflect.get_graph().draw_mermaid_png()))
print(full_graph.get_graph().draw_ascii(), end="\n\n")


    +-----------+    
    | __start__ |    
    +-----------+    
          *          
          *          
          *          
    +-----------+    
    | summarize |    
    +-----------+    
          *          
          *          
          *          
+------------------+ 
| tasks_structured | 
+------------------+ 
          *          
          *          
          *          
    +---------+      
    | reflect |      
    +---------+      
          *          
          *          
          *          
    +---------+      
    | __end__ |      
    +---------+      



In [47]:

final_state = full_graph.invoke({"transcript": transcript})

Reflection found issues (attempt 1):
 - The due_date "After this call" for the first task is not grounded in the transcript (no evidence on L12).
Instructions: ~ title=Draft marketing note about product launch assignee=B field=due_date -> null
Repair attempt 1 completed. New Tasks: tasks=[Task(title='Draft marketing note about product launch', assignee='B', description='Draft a short note to marketing using target date language, indicating subject to final QA approval.', due_date=None, status='InProgress'), Task(title='Send daily progress updates', assignee='C', description='Send daily progress updates to stakeholders (A and B) to monitor for delays.', due_date='Daily', status='InProgress'), Task(title='Finalize client demo build for next Thursday', assignee='A', description='Ensure the demo build for the client demo next Thursday contains only internal features and no payment-related features.', due_date='Next Thursday', status='InProgress'), Task(title='Prepare clean demo build by We

In [48]:
final_state = full_graph.invoke({"transcript": follow_up_transcript})

Persisting (semantic upsert)…
  ├─ UPDATED → send-daily-progress-updates [status=InProgress] (score=0.797)
  ├─ CREATED → lead-client-demo-walkthrough [status=ToDo] (score=0.654)
  ├─ CREATED → kick-off-full-regression-test [status=ToDo] (score=0.511)
  ├─ CREATED → share-regression-test-report [status=ToDo] (score=0.686)
  ├─ CREATED → book-stakeholder-risk-review [status=ToDo] (score=0.524)
  ├─ CREATED → draft-campaign-brief-one-pager [status=ToDo] (score=0.58)
  ├─ CREATED → provide-richer-checkout-error-logs [status=ToDo] (score=0.462)


## Wrap‑up and next steps

We began with a simple prompt and iteratively built a small, working agentic system that parses a messy meeting transcript and outputs a clean summary, a structured task list, and recorded decisions. Along the way you learned how to:
- Decompose a complex task into modular nodes (summary → tasks → decisions) and wire them together with LangGraph.
- Add a critic/repair loop to improve quality before persisting results.
- Persist results across runs using a lightweight memory store and a “semantic upsert” keyed on task titles.

This pattern—**decompose, validate, reflect, and persist**—is a reusable template for many agentic workflows.

### Stretch exercise: refine the semantic matcher

Our current semantic upsert matches tasks based only on their titles. As you saw earlier, this can still produce duplicates when two tasks have different wording but the same intent. For a challenge, think about how you might incorporate the task descriptions into the similarity search. Where in the code would you need to make changes? How would you decide when to fall back on the description? Try modifying your graph accordingly, re‑run it on the follow‑up transcript, and see if fewer duplicates remain. Let me know how it goes—or take it as a take‑home challenge!
